In [1]:
# Install required libraries
!pip install dvc requests pandas matplotlib


In [3]:
from pathlib import Path

# Create project structure
Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("results").mkdir(exist_ok=True)


In [5]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import time
from datetime import datetime
from pathlib import Path


In [7]:
def fetch_bitcoin_price():
    url = 'https://api.coingecko.com/api/v3/simple/price'
    params = {
        'ids': 'bitcoin',
        'vs_currencies': 'usd',
        'include_market_cap': 'true',
        'include_24hr_vol': 'true'
    }
    
    try:
        response = requests.get(url, params=params)
        if response.status_code == 200:
            data = response.json()
            return {
                'timestamp': datetime.now().isoformat(),
                'price_usd': data['bitcoin']['usd'],
                'market_cap': data['bitcoin']['usd_market_cap'],
                '24h_volume': data['bitcoin']['usd_24h_vol']
            }
        else:
            print(f"API Error: {response.status_code}")
            return None
    except Exception as e:
        print(f"Request Failed: {str(e)}")
        return None



In [ ]:
# ------------------------------------------------------------
# 1. Install required libraries in Google Colab
# ------------------------------------------------------------
!pip install pandas requests
!pip install dvc[gs]  # [gs] includes support for Google Drive remote if needed

# ------------------------------------------------------------
# 2. Clone your GitHub repository (OPTIONAL: if you're pushing to repo)
# ------------------------------------------------------------
# Replace with your own GitHub repo if needed
# !git clone https://github.com/your-username/your-repo.git
# %cd your-repo

# ------------------------------------------------------------
# 3. Set up Git and DVC
# ------------------------------------------------------------
!git init
!dvc init --no-scm  # Use --no-scm because we can't use full git integration in Colab easily

# ------------------------------------------------------------
# 4. Import required Python libraries
# ------------------------------------------------------------

import requests
import pandas as pd
from datetime import datetime
import os

# ------------------------------------------------------------
# 5. Function to fetch real-time Bitcoin price from CoinGecko
# ------------------------------------------------------------

def fetch_bitcoin_price():
    """
    Fetches the current price of Bitcoin in USD from the CoinGecko API.
    """
    url = 'https://api.coingecko.com/api/v3/simple/price'
    params = {'ids': 'bitcoin', 'vs_currencies': 'usd'}

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()

        price = data['bitcoin']['usd']
        timestamp = datetime.utcnow().isoformat()
        return {'timestamp': timestamp, 'price_usd': price}

    except Exception as e:
        print(f"Error fetching price: {e}")
        return None

# ------------------------------------------------------------
# 6. Function to save data to CSV
# ------------------------------------------------------------

def save_price_to_csv(data, filename='data/bitcoin_prices.csv'):
    """
    Appends the Bitcoin price data to a CSV file.
    Creates the file if it does not exist.
    """
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    df = pd.DataFrame([data])

    if os.path.exists(filename):
        df.to_csv(filename, mode='a', header=False, index=False)
    else:
        df.to_csv(filename, index=False)

# ------------------------------------------------------------
# 7. Function to track file with DVC (no Git commit needed in Colab)
# ------------------------------------------------------------

def track_with_dvc(filepath='data/bitcoin_prices.csv'):
    """
    Adds the CSV file to DVC tracking.
    """
    try:
        !dvc add {filepath}
        print("Tracked with DVC.")
    except Exception as e:
        print(f"Failed to track with DVC: {e}")

# ------------------------------------------------------------
# 8. Main logic
# ------------------------------------------------------------

price_data = fetch_bitcoin_price()

if price_data:
    print("Fetched data:", price_data)
    save_price_to_csv(price_data)
    track_with_dvc()
else:
    print("Failed to fetch data.")
